In [ ]:
import numpy as np
import pandas as pd
from pandas.api.types import is_datetime64_any_dtype
import os

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
date_Format = DateFormatter("%y-%m-%d")
%matplotlib inline
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LogNorm,SymLogNorm
from matplotlib.patches import Circle,Annulus
from astropy.visualization import ZScaleInterval
props = dict(boxstyle='round', facecolor="white", alpha=0.1)
#props = dict(boxstyle='round')
import matplotlib.colors as colors
import matplotlib.cm as cmx

import matplotlib.ticker                         # here's where the formatter is
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter,
                               AutoMinorLocator)

from matplotlib.gridspec import GridSpec


In [ ]:
# Remove to run faster the notebook
import ipywidgets as widgets
%matplotlib widget

In [ ]:
FILTER_COLORS = {
    "empty"       : "tab:gray",
    "BG40_65mm_1" : "darkblue",
    "OG550_65mm_1": "darkred",
    "FELH0600"    : "darkpurple",
}

DEFAULT_FILTER_COLOR = "purple"
DEFAULT_TARGET_COLOR ="lightgrey"
FILTER_MARKERS = {
    "empty"        : "o",
    "BG40_65mm_1"  : "^",
    "OG550_65mm_1" : "x",
    "FELH0600"     : "s",
}
DEFAULT_FILTER_MARKER = "."

LSST_FILTERS_ORDER = ["u", "g", "r", "i", "z", "y"]

LSST_FILTER_COLORS = {
    "u": "tab:blue",
    "g":  "tab:green",
    "r": "tab:red",
    "i": "tab:orange",
    "z": "tab:gray",
    "y": "black"
}
LSST_FILTER_COLORS = {
    "u": "#4DA6FF",  # bright blue
    "g": "#2ECC71",  # bright green
    "r": "#FF4D4D",  # bright red
    "i": "#FF9F1A",  # bright orange
    "z": "#9B59B6",  # bright purple
    "y": "#34495E",  # dark steel (lisible sur fond blanc)
}
def get_filter_color(filter_name):
    return FILTER_COLORS.get(filter_name, DEFAULT_FILTER_COLOR)


In [ ]:
#----------------
def plot_atmparam_vs_time(
    df,
    time_col,
    filter_col,
    param_col,
    param_err_col = None,
    # thresholds / bounds
    param_min_fig=None,
    param_max_fig=None,
    param_min_cut=None,
    param_max_cut=None,

    title_param = None,

    # display
    marker="+",
    lw=1.5,
    alpha=0.5,

    # datetime
    date_format="%y-%m-%d",

    # titres
    suptitle=None,

    # axes externes
    axs = None,
    figsize=(18, 6),
):
    """
    Trace param_col vs time

    Parameters
    ----------
    axs : tuple(matplotlib.axes.Axes), optional
        (ax1, ax2) créés à l'extérieur. Si None, la figure est créée ici.
    """

    if title_param is None:
        title_param=f"{param_col} vs time"


    data = df.copy()


    # ----------------------------
    # Gestion datetime (robuste)
    # ----------------------------
    if is_datetime64_any_dtype(data[time_col]):
        try:
            data[time_col] = data[time_col].dt.tz_convert(None)
        except TypeError:
            pass

    # Codage numérique des filtres (palette discrète stable)
    filter_cat = data[filter_col].astype("category")
    filter_codes = filter_cat.cat.codes
    filter_labels = filter_cat.cat.categories

    date_form = DateFormatter(date_format)

    # ----------------------------
    # Axes
    # ----------------------------

    if axs is None:
        fig, ax = plt.subplots(1, 1, figsize=figsize)
    else:
        ax = axs
        fig = ax.figure


    # ----------------------------
    # Plot per filter
    # ----------------------------

    filters = data[filter_col].unique()

    for f in filters:
        sub = data[data[filter_col] == f]

        if param_err_col == None:

            sc= ax.scatter(
                sub[time_col],
                sub[param_col],
                color=get_filter_color(f),  # <- couleur fixe
                marker=marker,
                lw=lw,
                alpha=alpha,
                label=f  # pour la légende
            )
        else:
            ax.errorbar(
                sub[time_col],
                sub[param_col],
                yerr=sub[param_err_col],
                fmt=marker,
                color=get_filter_color(f),
                elinewidth=lw,
                capsize=0,
                alpha=alpha,
                label=f,
            )


    ax.legend(title=filter_col, ncol=len(filters))



    if param_min_fig is not None and param_max_fig is not None:
        ax.set_ylim(param_min_fig, param_max_fig)

    if param_min_cut is not None:
        ax.axhline(param_min_cut, ls="-.", c="k")
    if param_max_cut is not None:
        ax.axhline(param_max_cut, ls="-.", c="k")

    # ----------------------------
    # Labels, grid
    # ----------------------------


    #handles, _ = sc1.legend_elements(prop="colors", alpha=alpha)
    #ax1.legend(
    #    handles,
    #    filter_labels,
    #    title=filter_col,
    #    ncols=len(filter_labels),
    #)

    ax.set_ylabel(f"{param_col}")
    ax.set_title(title_param)
    ax.grid(True, alpha=0.3)


    # ----------------------------
    # Time axis
    # ----------------------------

    ax.xaxis.set_major_formatter(date_form)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")


    # ----------------------------
    # Ticks on all sides
    # ----------------------------
    ax.minorticks_on()

    ax.tick_params(
        axis="x",
        which="both",
        top=True,        # show ticks on top
        bottom=True,     # keep bottom ticks
    )
    ax.tick_params(
        axis="y",
        which="both",
        left=True,       # show ticks on left
        right=True,
        labelleft=True,
        labelright=True, # o   # optional: set True if you want right ticks too
    )

    # ----------------------------
    # Global title
    # ----------------------------

    if suptitle:
        fig.suptitle(suptitle)

    if ax is None:
        fig.tight_layout()

    return fig, ax

#--------------------

In [ ]:
TOPDIR = "/sdf/group/rubin/user/cravoux/spectractor_runs/"

In [ ]:
!ls TOPDIR

In [ ]:
!ls /sdf/group/rubin/user/cravoux/spectractor_runs/

In [ ]:
atmfilename = os.path.join(TOPDIR,"auxtel_atmosphere_feb26_gaiaspec_calspecgaiatarget_calspecthroughput.npy")

In [ ]:
specdata = np.load(atmfilename, allow_pickle=True)
df_spec = pd.DataFrame(specdata)

In [ ]:
df_spec["Time"] = pd.to_datetime(df_spec["DATE-OBS"],utc=True)

In [ ]:
df_spec["nightObs"] = df_spec.apply(lambda x: x['id']//100_000, axis=1)
df_spec["seq_num"]  = df_spec["id"] % 100_000

In [ ]:
df_spec = df_spec[df_spec["FILTER"].isin(["empty","OG550_65mm_1"]) ]

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(20,6))
df_spec.plot(x="Time",y="PWV [mm]_rum",marker="+",lw=0,ax=ax)
ax.grid()

In [ ]:
plot_atmparam_vs_time(
    df_spec,
    "Time",
    "FILTER",
    "PWV [mm]_rum")